# main.cpp 的逻辑

该函数是 **QuoteFactory（数据落地工厂）的主程序入口**。它负责启动整个应用程序，解析启动参数，初始化日志和核心环境，调用全局初始化逻辑，并维持主线程运行直到接收到退出信号。

* **1. 参数解析**
  * 使用 `cppcli::Option` 解析命令行参数：
  * `-c` / `--config`：指定主配置文件路径（默认为 `dtcfg.yaml`）。
  * `-l` / `--logcfg`：指定日志配置文件路径（默认为 `logcfgdt.yaml`）。
  * `-h` / `--help`：显示帮助信息。
* **2. 日志系统初始化**
  * 根据命令行参数确定日志配置文件路径。
  * 调用 `WTSLogger::init` 初始化日志系统，确保后续运行中的信息能被正确记录。
* **3. 平台特定设置 (Windows)**
  * 如果在 Windows 环境下编译 (`_MSC_VER`)：
  * 设置 CRT 调试报告模式。
  * 设置错误模式和中止行为。
  * 获取主线程 ID，用于控制台事件处理。
  * 设置控制台控制处理函数 `ConsoleCtrlhandler`，用于捕获关闭窗口事件。
  * 启用 `CMiniDumper`，以便在程序崩溃时生成 Dump 文件用于调试。
* **4. 信号处理钩子安装**
  * 定义一个布尔标志 `bExit`，用于控制主循环的退出。
  * 调用 `install_signal_hooks` 安装信号处理函数：
    * **错误回调**：打印错误日志。
    * **退出回调**：当接收到 `SIGINT` (Ctrl+C) 或 `SIGTERM` 信号时，将 `bExit` 置为 `true`，通知程序准备退出。
* **5. 配置文件加载准备**
  * 确定主配置文件路径（优先使用命令行参数，否则使用默认值）。
  * 检查配置文件是否存在，若不存在则打印错误信息并退出程序。
* **6. 全局初始化 (`initialize`)**：严格按照依赖顺序依次初始化各个全局组件：
  * **1. 环境配置**：调用 `WtHelper::set_module_dir` 设置当前模块目录，确保动态库加载路径正确。
  * **2. 加载主配置**：读取 `dtcfg.yaml`（或用户指定的文件）。
  * **3. 加载基础数据**（通过 `g_baseDataMgr`）：
    * **交易时段** (`session`)：加载交易时间模板。
    * **品种信息** (`commodity`)：加载品种定义。
    * **合约信息** (`contract`)：加载具体合约列表。
    * **节假日** (`holiday`)：加载节假日列表。
  * **4. 加载主力规则**（通过 `g_hotMgr`）：
    * 加载主力合约切换规则 (`hot`)、次主力规则 (`second`) 以及自定义规则 (`rules`)。
  * **5. 初始化广播器**：
    * **共享内存广播**：如果有配置 `shmcaster`，初始化 `g_shmCaster` 并注册到数据管理器。
    * **UDP 广播**：如果有配置 `broadcaster`，初始化 `g_udpCaster` 并注册到数据管理器。
  * **6. 初始化状态监控**（条件分支）：
    * 检查配置中的 `allday`（全天候模式）标志。
    * 如果**不是**全天候模式，初始化 `g_stateMon`，用于根据交易时间段控制行情接收状态。
  * **7. 初始化数据管理器**：
    * 调用 `initDataMgr`（内部调用 `g_dataMgr.init`），传入 Writer 配置（存储路径、格式等）。
    * 如果是标准模式，将状态监控器 `g_stateMon` 绑定到数据管理器；全天候模式则传入 NULL。
  * **8. 初始化指数工厂**：
    * 如果有 `index` 配置，加载指数配置文件并初始化 `g_idxFactory`，用于实时计算衍生指数。
  * **9. 加载行情解析器**：
    * 读取 `parsers` 配置（支持独立文件或内嵌配置）。
    * 调用 `initParsers`：遍历配置，动态创建 `ParserAdapter` 实例，自动生成解析器 ID，并添加到 `g_parsers` 管理器中。
  * **10. 启动组件**：
    * 调用 `g_parsers.run()`：启动所有行情接入适配器，开始接收数据。
    * （非全天候模式）调用 `g_stateMon.run()`：启动状态监控线程。
* **7. 主循环 (运行保持)**
  * 进入 `while (!bExit)` 循环：
    * 只要没有接收到退出信号，主线程就保持运行。
    * 每次循环休眠 10 毫秒 (`std::this_thread::sleep_for`)，以避免占用过多 CPU 资源。
    * *注：实际的数据接收和处理工作由 `initialize` 启动的各个子线程（如 Parser 线程、UDP 广播线程等）在后台并发执行。*
* **8. 程序退出**
  * 当 `bExit` 变为 `true` 时跳出循环，返回 0，程序正常结束。
```cpp
/**
 * @brief QuoteFactory程序主入口函数
 * @param argc 命令行参数个数
 * @param argv 命令行参数数组
 * @return 程序退出码（0表示正常退出）
 */
int main(int argc, char* argv[])
```